# 04. Matplotlib

Неделя 3.

**Правило блокнота:** только объектный интерфейс — `fig, ax = plt.subplots()`, затем методы `ax.plot(...)`, `ax.set_xlabel(...)`. Функции вида `plt.plot`, `plt.xlabel` не используем (кроме `plt.subplots`, `plt.show`, `plt.close`).

**Как работать с блокнотом**
- Сначала решаешь сам, без ИИ. В задачах «предскажи» сначала пишешь ответ в комментарий, потом запускаешь.
- Непонятное поведение — прогони фрагмент в [pythontutor.com](https://pythontutor.com/) и посмотри на стрелки «имя → объект».
- Ячейки Jupyter можно запускать в любом порядке, и состояние (переменные) копится между ними. Это источник «у меня работало». После всех выполненных заданий: **Kernel → Restart & Run All** — блокнот должен пройти сверху вниз без ошибок.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)
print(matplotlib.__version__, "| backend:", matplotlib.get_backend())

## 1. Figure и Axes

- **`Figure`** — весь холст (окно или файл). Размер задаётся в **дюймах**: `figsize=(6, 3)`.
- **`Axes`** — одна система координат с осями, подписями, легендой. На одной `Figure` может быть несколько `Axes`.
- Каждая нарисованная вещь (линия, точки, текст) — объект-«художник» (`Artist`), который принадлежит конкретным `Axes`.

**Почему объектный интерфейс.** `plt.plot(...)` рисует на «текущих» `Axes`, которые matplotlib запоминает в глобальном состоянии. Когда графиков несколько или код разнесён по функциям, становится неясно, куда именно попадёт линия. `ax.plot(...)` всегда рисует ровно туда, куда ты указал.

**Jupyter и скрипты ведут себя по-разному:**
- в Jupyter (backend `inline`) фигура показывается автоматически в конце ячейки и **закрывается** после показа;
- в скрипте фигура появляется только по `plt.show()`, который **блокирует** выполнение, пока окно не закрыто. После закрытия окна фигура может быть уже уничтожена, поэтому `fig.savefig(...)` вызывают **до** `plt.show()`.

In [ ]:
t = np.linspace(0, 2 * np.pi, 200)   # 200 точек от 0 до 2π включительно
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t, np.sin(t), label="sin")
ax.plot(t, np.cos(t), "--", label="cos")          # "--" — штриховая линия
ax.set(xlabel="t, рад", ylabel="амплитуда", title="Пример")
ax.grid(alpha=0.3)                                # alpha — прозрачность от 0 до 1
ax.legend()                                       # подписи берутся из label=
plt.show()
print(type(fig).__name__, type(ax).__name__, "| линий на ax:", len(ax.lines))

### Задача 1. Сигнал и сглаживание

1. Сгенерируй `t` — 500 точек на отрезке [0, 4π] и сигнал `y = sin(t) + шум`, где шум — нормальный с σ = 0.3 (`rng.normal(0, 0.3, size=...)`).
2. Посчитай скользящее среднее с окном k = 25 функцией `moving_avg` из блокнота 03 (скопируй её сюда).
3. На **одних** `Axes`: сырые точки через `ax.scatter` (мелкие, `s=5`, полупрозрачные, `alpha=0.4`) и скользящее среднее — линией. Для сравнения тонкой линией нарисуй чистый `sin(t)`.
4. Подписи осей с единицами, заголовок, легенда.

**Тонкость, без которой график будет неправильным:** скользящее среднее короче исходного сигнала на `k − 1` точек. Если нарисовать его против `t[:len(ma)]`, линия сдвинется влево на половину окна. Каждое значение — это среднее окна `t[i : i+k]`, поэтому его нужно ставить в **центр** окна: `t[k//2 : k//2 + len(ma)]` (для нечётного k).

**Критерии приёмки:** сглаженная линия идёт поверх `sin(t)` без видимого сдвига по горизонтали; легенда не закрывает данные.

In [ ]:
# ваш код

### Задача 2. Гистограмма и теоретическая плотность

1. Сгенерируй 10 000 значений из нормального распределения N(μ = 2, σ = 0.5).
2. Нарисуй гистограмму `ax.hist(..., bins=50, density=True)`.
3. Поверх — теоретическую плотность нормального распределения линией. Формулу напиши сам, без scipy:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\, e^{-\frac{(x-\mu)^2}{2\sigma^2}}.$$

**Тонкости:**
- `density=True` нормирует гистограмму так, что **площадь** всех столбцов равна 1, — только тогда её можно сравнивать с плотностью. Высота столбца при этом может быть больше 1 (если столбцы узкие);
- `ax.hist` возвращает кортеж `(counts, edges, patches)`: `edges` — границы столбцов, их на одну больше, чем столбцов. Проверь, что `np.sum(counts * np.diff(edges))` равно 1;
- для линии плотности возьми свою сетку `x = np.linspace(...)`, а не исходные случайные точки (они не отсортированы, и линия превратится в «паутину»).

In [ ]:
# ваш код

### Задача 3. Сетка графиков

`fig, axs = plt.subplots(2, 2, figsize=(9, 7))` — здесь `axs` — массив NumPy формы `(2, 2)` из объектов `Axes`, обращение `axs[строка, столбец]`.

- `axs[0, 0]` — обычная линия (любая функция);
- `axs[0, 1]` — `errorbar` для 10 точек с погрешностями (`yerr`) и маркерами; `capsize=3` рисует «усики»;
- `axs[1, 0]` — `imshow` случайной матрицы 50×50 с цветовой шкалой;
- `axs[1, 1]` — `exp(x)` на x ∈ [0, 10] с логарифмической шкалой по y (`set_yscale("log")`). На ней экспонента должна выглядеть прямой — объясни почему.

В конце вызови `fig.tight_layout()`, чтобы подписи не налезали друг на друга.

**Тонкости:**
- `fig.colorbar` нужен объект, который вернул `imshow`: `im = axs[1, 0].imshow(...)`, затем `fig.colorbar(im, ax=axs[1, 0])`. Без явного `ax=` шкала может «отъесть» место не у тех `Axes`;
- `imshow` по умолчанию помещает элемент `[0, 0]` в **левый верхний** угол (как картинку), а не в левый нижний, как на графике. Параметр `origin="lower"` это меняет;
- на логарифмической шкале значения ≤ 0 не отображаются: проверь, что в данных их нет.

In [ ]:
# ваш код

### Задача 4. Рисунок для статьи

Функция `plot_measurements(x, y, yerr, path)`:
- график с погрешностями (`errorbar`), подписи осей **с единицами измерения**;
- размер фигуры 3.5 × 2.5 дюйма — ширина одной колонки в типичном журнале;
- шрифт 8 pt для всего текста на рисунке;
- сохранение в `path`; формат определяется расширением (`.pdf`, `.png`). Для PNG — `dpi=300`;
- функция **не** вызывает `plt.show()` и **закрывает** свою фигуру (`plt.close(fig)`).

**Тонкости:**
- шрифт задавай локально через `with plt.rc_context({"font.size": 8}):` — изменение глобального `plt.rcParams` «протечёт» во все последующие графики программы;
- `dpi` влияет только на растровые форматы (PNG). PDF и SVG — векторные: в них хранятся линии и текст, а не пиксели;
- `bbox_inches="tight"` обрезает лишние белые поля при сохранении;
- **зачем закрывать фигуру:** pyplot хранит ссылку на каждую созданную фигуру, пока её не закроют. В цикле по 1000 файлов незакрытые фигуры будут копиться в памяти, и после 20 штук matplotlib выдаст предупреждение `RuntimeWarning: More than 20 figures have been opened`.

In [ ]:
def plot_measurements(x, y, yerr, path):
    raise NotImplementedError

import os, tempfile
d = tempfile.mkdtemp()
x = np.arange(1, 11)
y = 2 * x + rng.normal(0, 1, 10)
figs_before = set(plt.get_fignums())          # номера открытых фигур до вызова
font_before = plt.rcParams["font.size"]
for ext in ("pdf", "png"):
    out = os.path.join(d, f"fig.{ext}")
    plot_measurements(x, y, np.full(10, 1.0), out)
    assert os.path.getsize(out) > 0
assert set(plt.get_fignums()) == figs_before, "функция оставила открытые фигуры"
assert plt.rcParams["font.size"] == font_before, "глобальный rcParams изменён — используй rc_context"
print("OK:", sorted(os.listdir(d)))

## Вопросы
1. Чем `Figure` отличается от `Axes`? Сколько `Axes` может быть на одной `Figure`?
2. Чем `plt.plot` отличается от `ax.plot` и почему в скриптах лучше второе?
3. Почему `savefig` нужно вызывать до `plt.show()` в скрипте?
4. Что делает `density=True` и почему высота столбца может быть больше 1?
5. Почему скользящее среднее нужно рисовать против центров окон?
6. Почему для публикаций предпочтительнее векторный формат (PDF/SVG)? На что влияет `dpi`?
7. Зачем закрывать фигуры и почему шрифт лучше менять через `rc_context`?
8. Мини-проект недели 4: CSV → очистка → статистика → графики. Какие функции из блокнотов 02–04 ты переиспользуешь?